# Patch Image

> Patch image size to (128x128)

In [ ]:
#| default_exp patching_image

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import numpy as np
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from fastcore.all import *
from fastcore.basics import *
import matplotlib as mpl
from typing import NewType, Union, Tuple, List
from tqdm.auto import tqdm
import shutil
from argparse import ArgumentParser
import yaml
from scipy.ndimage import label

In [ ]:
#| export
custom_lib_path = Path("/home/ai_warstein/homes/goni/custom_libs")
sys.path.append(str(custom_lib_path))
CURRENT_DIR = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test')
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
NORMAL_FRONT_PIN = Path(r'/home/ai_sintercra/homes/hasan/easy_front/easy-front-pin-detection')
import sys
sys.path.append(str(CURRENT_DIR))
sys.path.append(str(CV_TOOLS))
sys.path.append(str(NORMAL_FRONT_PIN))

In [ ]:
from new_test.mask_generation_patch_image125 import *

In [ ]:
# Test image path

In [ ]:
im_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/Incoming_1B_Loetstift/Incoming_1B_Loetstift_unzip/main_im2_cropped_masks/time_11_04_21_val_frGrnd0.9498_epoch_200.h5/failed/missing/images')
CONFIG_PATH = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test/configs/config_test_patch_256.yaml')

In [ ]:
with open (CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
def split_image_to_patches(
        x, 
        patch_size=128,
        overlap=32
        ):
    '''
    Split the image into overlapping patches.
    
    Args:
        x: Input tensor with shape [batch, height, width, channels]
        patch_size: Size of each square patch
        overlap: Number of pixels to overlap between adjacent patches
    
    Returns:
        Tensor of patches with shape [num_patches, patch_size, patch_size, channels]
    '''
    # Extract dimensions
    batch_size, height, width, channels = x.shape
    
    # Calculate stride (distance between patch starts)
    stride = patch_size - overlap
    
    # Calculate number of patches in each dimension
    num_patches_h = (height - overlap) // stride + (1 if (height - overlap) % stride != 0 else 0)
    num_patches_w = (width - overlap) // stride + (1 if (width - overlap) % stride != 0 else 0)
    
    # Calculate padding to ensure we cover the full image
    pad_height = max(0, stride * num_patches_h + overlap - height)
    pad_width = max(0, stride * num_patches_w + overlap - width)
    
    # Apply padding
    x_padded = tf.pad(x, [[0, 0], [0, pad_height], [0, pad_width], [0, 0]], mode='CONSTANT')
    
    # Use tf.image.extract_patches which is ONNX compatible
    patches = tf.image.extract_patches(
        x_padded,
        sizes=[1, patch_size, patch_size, 1],
        strides=[1, stride, stride, 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )
    
    # Reshape to get individual patches
    patch_dims = tf.shape(patches)
    patches_reshaped = tf.reshape(patches, [batch_size * patch_dims[1] * patch_dims[2], patch_size, patch_size, channels])
    
    return patches_reshaped

In [ ]:
sm_im_path = im_path.ls()[0]


In [ ]:
tf_img = read_normalize(str(sm_im_path), config)

In [ ]:
tf_img.shape

In [ ]:
sm_im_path

In [ ]:
patches_ = split_image_to_patches(tf_img[None,...],patch_size=256)

In [ ]:
patches_.shape

# TODO

- [ ] Add padding
- [ ] patch image with overlap(25-30%)
- [ ] Reconstruct images with patches (mean , max) criteria


In [ ]:
patches_shape = tf.shape(patches_)
num_total_patches = patches_shape[0]
channels = patches_shape[3]
num_total_patches, channels

In [ ]:
patch_size = 256
tf.Assert(patches_shape[1] == patch_size, ["Patch height mismatch."])
tf.Assert(patches_shape[2] == patch_size, ["Patch width mismatch. "])

In [ ]:
original_height, original_width = 1632, 1152
# Ensure original dimensions are tensors for graph mode
original_height = tf.convert_to_tensor(original_height, dtype=tf.int32)
original_width = tf.convert_to_tensor(original_width, dtype=tf.int32)

In [ ]:
overlap = int(patch_size*.30)
stride = patch_size - overlap
print(f'stride = {stride}')
tf.Assert(stride > 0, ["Stride must be positive."])

In [ ]:
 # --- Calculate Padded Dimensions and Grid Size (Mirroring the patching logic) ---
# Re-calculate the required padded size based on original dims, patch_size, stride
# This ensures we create a canvas of the correct size that the patches fit into.
num_strides_h = tf.cast(
    tf.math.ceil(
        tf.math.maximum(
            0.0, tf.cast(
                original_height - patch_size, tf.float32) / tf.cast(stride, tf.float32))), tf.int32)
num_strides_w = tf.cast(tf.math.ceil(tf.math.maximum(0.0, tf.cast(original_width - patch_size, tf.float32) / tf.cast(stride, tf.float32))), tf.int32)

In [ ]:
num_strides_h, num_strides_w

In [ ]:
required_height = num_strides_h * stride + patch_size
required_width = num_strides_w * stride + patch_size
required_height, required_width

In [ ]:
#| eval: false
im_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped')
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_office_test.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
model = load_model(config)

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5/failed/missing/images')
overlay_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5/failed/missing/overlay')
mis_real_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5/failed/missing/missing_pin_real/images')
mis_mask_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5/failed/missing/missing_pin_real/only_masks')
mis_img_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5/failed/missing/missing_pin_real/only_images')
##############################################

im_name = Path(overlay_path.ls()[2]).name

#shutil.copyfile(f'{im_path}/{im_name}', f'{mis_real_path}/{im_name}')

In [ ]:
Path((f'{mis_img_path}/{[i for i in  (set(get_name_(mis_img_path.ls()))) if i not in set(get_name_(mis_mask_path.ls())) ][0]}')).unlink()

In [ ]:
i

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_start20240516_unzip/main_im2_cropped')
img = read_img(im_path.ls()[108])
print(f'file_name = {Path(im_path.ls()[100]).name}')
img = np.expand_dims((img/255.0).astype(np.float32), axis=-1)[None,...]
print(img.shape)
msk = model.predict(img)
show_(msk[0]>0.7)

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped')

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_23_32_06_val_frGrnd0.9556_epoch_195.h5/min_contour_area_300/failed/additional/images')
im1_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im1_cropped')
im_file = im_path.ls()[1]
name_ = im_file.name.replace('image2', 'image1')
im1 = read_img(im1_path/name_)

In [ ]:
def im1_im2(
    im1_path:Union[str, Path, None],
    im2_path:Union[str, Path, None],
    name:str=None
    ):
    if name is None:
        fn = np.random.choice(im2_path.ls(),1)[0]
        name_ = Path(fn).name
    else:
        name_ = name

    im1_name = name_.replace('image2', 'image1')

    im2_img = read_img(f'{im2_path}/{name_}')
    im1_img = read_img(f'{im1_path}/{im1_name}')
    return im2_img, im1_im



In [ ]:
im1_path = Path(r'/home/ai_sintercra.work/VF_test_unzip/_unzip/main_im1_cropped')

In [ ]:
im2_img, im1_img = im1_im2(
    im1_path=im1_path,
    im2_path=im_path
)

In [ ]:
#| export
from cv_tools.core import *
OpencvImage = NewType('OpencvImage', np.ndarray)

In [ ]:
from private_easy_pin_detection.mask_generation_patch_image125 import *

In [ ]:
image1.shape

In [ ]:
def reconstruct_from_patches(
        patches:tf.Tensor, 
        original_shape:tuple[int, int], 
        patch_size:int=128
        ):
    ' Reconstruction of the image from patches'

    # Assuming patches are square and the batch dimension is flat
    num_patches_per_row = tf.math.ceil(tf.cast(original_shape[1], float) / patch_size)
    num_patches_per_col = tf.math.ceil(tf.cast(original_shape[2], float) / patch_size)
    
    num_patches_per_row = tf.cast(num_patches_per_row, tf.int32)
    num_patches_per_col = tf.cast(num_patches_per_col, tf.int32)
    patch_size = tf.cast(patch_size, tf.int32)
    # Reshape to grid
    patches = tf.reshape(patches, (
        -1, 
        num_patches_per_row, 
        num_patches_per_col, 
        patch_size, 
        patch_size, 
        original_shape[-1]))
    patches = tf.transpose(patches, [0, 1, 3, 2, 4, 5])
    patches = tf.reshape(
        patches, 
        (
            -1, 
            tf.cast(num_patches_per_row * patch_size, tf.int32),
            tf.cast(num_patches_per_col * patch_size, tf.int32),
            tf.cast(original_shape[-1], tf.int32)
        ))
    
    # Crop to original dimensions
    return patches[:, :original_shape[1], :original_shape[2], :]


In [ ]:
im2_img, im1_img = im1_im2(
    im2_path=im_path,
    im1_path=im1_path
)

In [ ]:
im2_img.shape, im1_img.shape, np.max(im2_img)

In [ ]:
show_(im2_img)

In [ ]:
def frm_patch_mdl_to_one_img_mdl(
    patch_model:tf.keras.Model, 
    config:dict,
    desired_shape:tuple[int, int],
    
    
):
    inputs_ = tf.keras.layers.Input(
    shape= tuple(config['model']['input_shape'])
    )
    patches = split_image_to_patches(inputs_, config['dataset']['patch_size'])
    old_model = load_model(config)
    patch_pred = old_model(patches)
    reconstructed_image=reconstruct_from_patches(
        patch_pred, 
        original_shape=desired_shape, 
        patch_size=config['dataset']['patch_size'])
    prd = reconstructed_image > config['model']['threshold']
    return tf.keras.models.Model(
    inputs=[inputs_],
    outputs=[prd])
    

###  Loading Patch Model

In [ ]:
#im_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped')
im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_2608/missing_2608_unzip/main_im2_cropped')
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_test_patch_256_input_shape1152_1632.yml')
with open(config_path, 'r') as f:
    patch_config = yaml.safe_load(f)
patch_model = load_model(patch_config)

In [ ]:
one_image_mdl = frm_patch_mdl_to_one_img_mdl(
    patch_model=patch_model,
     config=patch_config,
    desired_shape=(1, 1152, 1632, 1), 
   )

In [ ]:
one_image_mdl.summary()

In [ ]:
sm_fn = Path(im_path.ls()[1])
sm_img = read_img(sm_fn)

In [ ]:
sm_img = (sm_img/256.0)[None, ..., None]
sm_img = sm_img.astype(np.float32)
sm_pred = one_image_mdl.predict(sm_img)

In [ ]:
show_(sm_pred[0,:,:,0])

In [ ]:
show_(sm_pred[0,:,:,0])

### checking each patch prediction

In [ ]:
patches = split_image_to_patches(sm_img, patch_size=128)

In [ ]:
patch_pred = patch_model.predict(patches)

In [ ]:
patch_pred.shape

In [ ]:
def get_n_pxl(binary_msk):
    lbld_msk, n_ftr = label(new_img_)
    for i in range(1, n_ftr):
        component_mask = (lbld_msk==i)
        n_pixel = np.sum(component_mask)
        if n_pixel >10:
            print(f'part {i} has {n_pixel}')
        else:
            print(f'part {i} has {n_pixel}')
    

In [ ]:
show_(new_img_)

In [ ]:
patch_pred_n_ = list(filter(lambda y: np.max(np.array(y> 0.5, dtype=np.uint8))==1, patch_pred))

In [ ]:
len(patch_pred_n_)

In [ ]:
show_(patch_pred_n_[4])

In [ ]:
from cv_tools.core import *

In [ ]:
b_m =np.array(patch_pred_n_[4] > 0.5, dtype=np.uint8)

In [ ]:
cntrs_ = find_contours_binary(b_m)

In [ ]:
len(cntrs_)

In [ ]:
for i in cntrs_ : 
    a_ = cv2.contourArea(i)
    print(a_)

In [ ]:
for i in patch_pred:
    new_img_ = np.array(i >= 0.5, dtype=np.uint8)
    labeled_msk, n_feature = label(new_img_)
     
    max_ = np.max(new_img_)
    if max_ == 1:
        get_n_pxl(new_img_)
        show_(new_img_)

In [ ]:
split_image_to_patches()

In [ ]:
patches = split_image_to_patches(
    image2, 
    patch_size=256
)

In [ ]:
reconstruc_from_patches

In [ ]:
old_m_p = Path(Path(save_path).parent, 'time_21_18_40_val_frGrnd0.9548_epoch_198.h5')

In [ ]:
inputs = tf.keras.layers.Input(
    shape=tuple(config['model']['input_shape']))

image2, image1 = inputs[:,:,:,0:1], inputs[:, :, :,1:2]
print(image2.shape,image1.shape)
patches = split_image_to_patches(
    image2, 
    patch_size=config['dataset']['patch_size']
)
old_model = tf.keras.models.load_model(f'{old_m_p}',compile=False)
patch_pred = old_model(patches)

pred = reconstruct_from_patches(
    patches=patch_pred,
    original_shape=(1,1152, 1632,1),
    patch_size=config['dataset']['patch_size']

)

msk = tf.cast(pred >= 0.5, tf.float32)

output_ = tf.keras.layers.Multiply()([msk, image1])

new_model = tf.keras.models.Model(
    inputs=[inputs],
    outputs=[output_]
)

In [ ]:
save_path = f"{config['model']['save_path']}/patch_two_images_t_0.5.keras"

In [ ]:
save_path = f"{config['model']['save_path']}/patch_two_images_t_0.5.keras"

In [ ]:
tf.keras.models.save_model(
    new_model,
    save_path
)

In [ ]:
tf.keras.models.save_model(
    new_model,
    save_path
)

In [ ]:
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_test_two_image_with_patch.yml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
m = load_model(config)

In [ ]:
save_path = f"{config['model']['save_path']}/patch_two_images.keras"

In [ ]:
trn_im_path = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_images_patch_256_min_px_350')
trn_msk_path = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_masks_patch_256_min_px_350')

In [ ]:
def create_overlay_image(
        im_path:Union[str, Path],
        msk_path:Union[str,Path],
        overlay_mask_save_path:Union[str, Path],
        num_workers:int=8,
        pr_im_path:Union[str, Path]=None
    ):
    'Create a overlay mask from image and mask path'
    def _overlay_mask_for_single_image(name_):
        if pr_im_path is not None:
            pr_name = name_.replace("image2", "processed_image")
            pr_img = read_img(f"{pr_im_path}/{pr_name}")
            overlay_mask_border_on_image(
                im_path=f"{im_path}/{name_}",
                msk_path=f"{msk_path}/{name_}",
                new_img=pr_img,
                save_overlay_img_path=overlay_mask_save_path,
            )
        else:
            overlay_mask_border_on_image(
                im_path=f"{im_path}/{name_}",
                msk_path=f"{msk_path}/{name_}",
                save_overlay_img_path=overlay_mask_save_path,
            )
    if pr_im_path is not None:
        image_names = [Path(i).name for i in im_path.ls()]
    else:
        image_names = set(get_name_(im_path.ls())).intersection(
            set(get_name_(msk_path.ls()))
        )

    parallel(_overlay_mask_for_single_image, image_names, n_workers=num_workers)
    

In [ ]:
IMAGE2_SOURCE="/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped"
model_name="time_23_32_06_val_frGrnd0.9556_epoch_195_with_two_image.h5"
create_overlay_image(
    im_path=Path(IMAGE2_SOURCE),
    msk_path=Path(f"{IMAGE2_SOURCE}_masks/{model_name}"),
    overlay_mask_save_path=f"{IMAGE2_SOURCE}/overlay_masks_{model_name}",
)

In [ ]:
path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im1_cropped')

In [ ]:
from cv_tools.compress_and_filter import *

In [ ]:
convert_image_to_parquet_p(path,file_name_func=None,
    file_exts='.png',
    threadpool=False)

In [ ]:
#| hide
mpl.rcParams['image.cmap'] = 'gray'

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped')
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_test_two_image_with_patch.yml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
new_m = load_model(config)

In [ ]:
#model = load_model(config)

In [ ]:
im1_n = (im1/255.0)[None,...]

In [ ]:
pred[0].shape

In [ ]:
im1_n.shape

In [ ]:
model.summary()

In [ ]:
model = load_model(config)

In [ ]:
show_(im_file)

In [ ]:
config['dataset']['min_contour_area']=0

In [ ]:
model

In [ ]:
im_file = im_path.ls()[1]
img = read_normalize(im_file=f"{im_file}", config=config)
img_b = img[None, ...]
img_patchs = split_image_to_patches(
    img_b, patch_size=256
)
pred = model.predict(img_patchs)
pred = reconstruct_from_patches(
        patches=pred,
        original_shape=img_b.shape,
        patch_size=256
    )
#pred.shape
#prd_msk = pred[0]
#pred_mask = prd_msk > .9
#msk = pred_mask.numpy().astype(np.uint8)

In [ ]:
show_(((msk[:, :, 0] * im1)))

In [ ]:
np.min()

In [ ]:
show_(msk)

In [ ]:
areas = [cv2.contourArea(i) for i in cntrs]
sorted(areas)

In [ ]:

for idx, i in enumerate(img_patchs):
    new_  = i[None, ...]
    n_msk = model.predict(new_)
    n_msk_ = n_msk[0] > 0.9

    msk_img = (n_msk_ * 255).astype(np.uint8)
    cntrs_ = find_contours_binary(msk_img)
    if cntrs_:
        area_ = [cv2.contourArea(i) for i in cntrs_]
        max_area, min_area =  np.max(area_), np.min(area_)
    else: max_area, min_area = 0, 0

    new_im = (i * 255).numpy().astype(np.uint8)
    msk_im_to = concat_images([new_im, msk_img],rows=1,  cols=2, number=f'{idx}, min_area={min_area}, max_area={max_area}')
    show_(msk_im_to)




In [ ]:
msk = predict_msk_patch_125(
    model, 
    config=config, 
    patch_size=256,
    im_file=f"{im_path.ls()[0]}")

In [ ]:
show_(msk)

In [ ]:
show_(msk)

In [ ]:
msk = predict_msk_patch_125(
    model, 
    config=config, 
    patch_size=256,
    im_file=f"{im_path.ls()[0]}")

In [ ]:
show_(msk)

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_23_32_06_val_frGrnd0.9556_epoch_195.h5/min_contour_area_300/failed/additional/images')
sm_img = read_img(im_path.ls()[0])

In [ ]:
M:\Fail_start20240402\_unzip\main_im2_cropped_masks\time_23_32_06_val_frGrnd0.9556_epoch_195.h5\min_contour_area_300

In [ ]:
im_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240418_trn_val/trn_images')
msk_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240418_trn_val/trn_masks')

im_path.ls()

In [ ]:
#| export
def patchify_(
    img:OpencvImage,
    patch_size:int=128,
    msk:Union[OpencvImage,None]=None,
    stem_:str='image_name', # without extension of image name
	min_pixel:int=350,
    save_patch_path:Union[Path,None]=None,
    save_msk_patch:Union[Path,None]=None,

    ):
    'patch iamges and extra part will be zero'
    h, w = img.shape[:2]

    n_p_w = (w + patch_size  - 1 ) // patch_size 
    n_p_h = (h + patch_size  - 1 ) // patch_size


    padded_w = n_p_w  * patch_size
    padded_h = n_p_h  * patch_size
    padded_img = np.zeros((padded_h, padded_w), dtype=img.dtype)
    if msk is not None:
        padded_msk = np.zeros((padded_h, padded_w), dtype=msk.dtype)
        padded_msk[:h, :w] = msk



    if save_msk_patch is not None:
        Path(save_msk_patch).mkdir(exist_ok=True, parents=True)

    
    padded_img[:h, :w] = img



    patches = []
    b_img = np.zeros_like(img)
    for i in range(n_p_h):
        for j in range(n_p_w):

            x = j * patch_size
            y = i * patch_size
            patch = padded_img[y:y+patch_size, x:x+patch_size]
            patch_msk = padded_msk[y:y+patch_size, x:x+patch_size]
            if cv2.countNonZero(patch_msk) > 10:
                cntrs = find_contours_binary(msk)
                new_cntrs = list(filter(lambda x: cv2.contourArea(x)>=min_pixel, cntrs))
                # checking each pin to have specific pixel count
                # if all pin inside the mask has more than min_pixel then copy image and mask
                # checking whether number of pins which has mix pixel 
                # is equal to number of pins in the mask
                if len(new_cntrs) == len(cntrs):
                    if save_patch_path is not None:
                        save_patch_path.mkdir(exist_ok=True, parents=True)
                        cv2.imwrite(str(save_patch_path/f'{stem_}_{i}_{j}.png'), patch)
                    if save_msk_patch is not None:
                        cv2.imwrite(
                            str(
                                save_msk_patch/f'{stem_}_{i}_{j}.png'), 
                                padded_msk[y:y+patch_size, x:x+patch_size])
                        b_img[y:y+patch_size, x:x+patch_size] = img[y:y+patch_size, x:x+patch_size]
                    patches.append(patch)
    return patches, b_img


In [ ]:
#| export
def patchify_path(
    im_path:Union[Path, str],
    patch_size:int=128,
    save_im_path:Union[Path, str, None]=None,
    save_msk_path:Union[Path, str, None]=None,
    msk_path:Union[Path, str]=None,
	min_pixel:int=350,

  ):
  for i in tqdm(im_path.ls(), total=len(im_path.ls())):
    img = read_img(i)
    if msk_path is not None:
      msk = read_img(msk_path/i.name)
    else:
      msk = None
    parts, b_img = patchify_(
        img=img, 
        patch_size=patch_size,
        stem_=i.stem,
        msk=msk,
		min_pixel=min_pixel,
        save_patch_path=save_im_path,
        save_msk_patch=save_msk_path
        )

In [ ]:
#im_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/trn_images')
#msk_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/trn_masks')
save_im_path = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_images_patch_256_min_px_350')
save_msk_path = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_masks_patch_256_min_px_350')
#save_msk_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/CurrentTrainingData20240418_trn_val/trn_masks_patch_350')

patchify_path(
    im_path=im_path,
	patch_size=256,
	save_im_path=save_im_path,
	save_msk_path=save_msk_path,
	msk_path=msk_path,
	min_pixel=350
	)

In [ ]:
im_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240418_trn_val/val_images')
msk_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240418_trn_val/val_masks')

save_im_path = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/val_images_patch_256_min_px_350')
save_msk_path = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/val_masks_patch_256_min_px_350')

patchify_path(
    im_path=im_path,
	patch_size=256,
	save_im_path=save_im_path,
	save_msk_path=save_msk_path,
	msk_path=msk_path,
	min_pixel=350
	)

In [ ]:
#| export
def get_common_files(im_path, msk_path):
    im_files = get_name_(im_path.ls())
    print(f' Number of images: {len(im_files)}')
    msk_files = get_name_(msk_path.ls())
    print(f' Number of masks: {len(msk_files)}')
    common_files = set(im_files).intersection(set(msk_files))
    print(f' Number of common files: {len(common_files)}')
    return common_files

In [ ]:
#| export
def min_pixel_msk(
        im_src:Union[Path, str],
        im_dst:Union[Path, str],
        msk_src:Union[Path, str],
        msk_dst:Union[Path, str],
        min_pixel:int=300
            ):
    'Search for mask and if mask has more than pixel then copy image and mask'

    common_files = get_common_files(im_src, msk_src)

    for i in tqdm(common_files, total=len(common_files)):

        msk = read_img(f'{msk_src}/{i}')
        pxl_cnt  = cv2.countNonZero(msk)
        if pxl_cnt >min_pixel:
            shutil.copyfile(f'{im_src}/{i}', f'{im_dst}/{i}')
            shutil.copyfile(f'{msk_src}/{i}', f'{msk_dst}/{i}')

In [ ]:
#| export
def min_pixel_msk_each_pin(
        im_src:Union[Path, str],
        im_dst:Union[Path, str],
        msk_src:Union[Path, str],
        msk_dst:Union[Path, str],
        min_pixel:int=300
            ):
    'Search for mask and if mask has more than pixel then copy image and mask'

    common_files = get_common_files(im_src, msk_src)

    for i in tqdm(common_files, total=len(common_files)):

        msk = read_img(f'{msk_src}/{i}')
        if cv2.countNonZero(msk) > 10:
            cntrs = find_contours_binary(msk)
            #for cnt in cntrs:
                #pa_ = cv2.contourArea(cnt)
                #print(pa_)
            new_cntrs = list(filter(lambda x: cv2.contourArea(x)>=min_pixel, cntrs))

            # checking each pin to have specific pixel count
            # if all pin inside the mask has more than min_pixel then copy image and mask
            # checking whether number of pins which has mix pixel 
            # is equal to number of pins in the mask
            if len(new_cntrs) == len(cntrs):


                shutil.copyfile(f'{im_src}/{i}', f'{im_dst}/{i}')
                shutil.copyfile(f'{msk_src}/{i}', f'{msk_dst}/{i}')

In [ ]:
im_src=Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_images_100')
im_dst = Path(im_src.parent, 'trn_images_400')
msk_src = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_masks_100')
msk_dst = Path(msk_src.parent, 'trn_masks_400')
im_dst.mkdir(exist_ok=True, parents=True)
msk_dst.mkdir(exist_ok=True, parents=True)

min_pixel = 400

In [ ]:
msk_dst

In [ ]:
min_pixel_msk(
    im_src=im_src,
    im_dst=im_dst,
    msk_src=msk_src,
    msk_dst=msk_dst,
    min_pixel=min_pixel
)

# patching and minimum pixel count in each patch in patch

In [ ]:
im_src=Path(r'/home/i4a_dev/data/projects/easy_pin_detection/val_images_100')
im_dst = Path(im_src.parent, 'val_images_400')
msk_src = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/val_masks_100')
msk_dst = Path(msk_src.parent, 'val_masks_400')
im_dst.mkdir(exist_ok=True, parents=True)
msk_dst.mkdir(exist_ok=True, parents=True)

min_pixel = 400

In [ ]:
im_src=Path(r'/home/ai_easypid.work/three_channel_ds/trn_sc_im2/images_patch')
im_dst = Path(im_src.parent, 'trn_images_400')
im_dst_each = Path(im_src.parent, 'trn_images_350_each')
msk_src = Path(r'/home/ai_easypid.work/three_channel_ds/trn_sc_im2/masks_patch')
msk_dst = Path(msk_src.parent, 'trn_masks_400')
msk_dst_each = Path(msk_src.parent, 'trn_masks_350_each')
im_dst.mkdir(exist_ok=True, parents=True)
msk_dst.mkdir(exist_ok=True, parents=True)
im_dst_each.mkdir(exist_ok=True, parents=True)
msk_dst_each.mkdir(exist_ok=True, parents=True) 

In [ ]:
im_dst_each

In [ ]:
im_src=Path(r'/home/ai_easypid.work/three_channel_ds/val_sc_im2/images_patch')
im_dst = Path(im_src.parent, 'val_images_400')
im_dst_each = Path(im_src.parent, 'val_images_350_each')
msk_src = Path(r'/home/ai_easypid.work/three_channel_ds/val_sc_im2/masks_patch')
msk_dst = Path(msk_src.parent, 'val_masks_400')
msk_dst_each = Path(msk_src.parent, 'val_masks_350_each')
im_dst.mkdir(exist_ok=True, parents=True)
msk_dst.mkdir(exist_ok=True, parents=True)
im_dst_each.mkdir(exist_ok=True, parents=True)
msk_dst_each.mkdir(exist_ok=True, parents=True)

In [ ]:
min_pixel= 350
min_pixel_msk_each_pin(
    im_src=im_src,
    im_dst=im_dst_each,
    msk_src=msk_src,
    msk_dst=msk_dst_each,
    min_pixel=min_pixel
)

# In case of gray scale image

### Validation

In [ ]:
im_src=Path(r'/home/i4a_dev/data/projects/easy_pin_detection/val_images')
im_dst = Path(im_src.parent, 'val_images_350_each')
msk_src = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/val_masks')
msk_dst = Path(msk_src.parent, 'val_masks_350_each')
im_dst.mkdir(exist_ok=True, parents=True)
msk_dst.mkdir(exist_ok=True, parents=True)

In [ ]:
min_pixel= 350
min_pixel_msk_each_pin(
    im_src=im_src,
    im_dst=im_dst,
    msk_src=msk_src,
    msk_dst=msk_dst,
    min_pixel=min_pixel
)

In [ ]:
### Training

In [ ]:
im_src=Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_images_100')
im_dst = Path(im_src.parent, 'trn_images_350_each')
msk_src = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_masks_100')
msk_dst = Path(msk_src.parent, 'trn_masks_350_each')
im_dst.mkdir(exist_ok=True, parents=True)
msk_dst.mkdir(exist_ok=True, parents=True)

In [ ]:
min_pixel= 350
min_pixel_msk_each_pin(
    im_src=im_src,
    im_dst=im_dst,
    msk_src=msk_src,
    msk_dst=msk_dst,
    min_pixel=min_pixel
)

In [ ]:
from functools import partial

In [ ]:

min_pixesl_msk_p = partial(
    min_pixel_msk_each_pin, 
    im_src=im_src, 
    im_dst=im_dst, 
    msk_dst=msk_dst,
    min_pixel=250)

In [ ]:
min_pixel_msk(
    im_src=im_src,
    im_dst=im_dst,
    msk_src=msk_src,
    msk_dst=msk_dst,
    min_pixel=min_pixel
)

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
        '--im_path',
        type=str,
        required=True,
        help='Path of the image'
    )
    parser.add_argument(
        '--msk_path',
        type=str,
        required=True,
        help='Path of the mask'
    )
    parser.add_argument(
        '--min_pixel',
        type=int,
        required=True,
        help='Minimum pixel count'
    )
    parser.add_argument(
        '--patch_size',
        type=int,
        default=128,
        help='Patch size'
    )
    parser.add_argument(
        '--save_im_path',
        type=str,
        default=None,
        help='Path to save image patches'
    )
    parser.add_argument(
        '--save_msk_path',
        type=str,
        default=None,
        help='Path to save mask patches'
    )
    return parser.parse_args()

In [ ]:
#| export
def main_():
    args = parse_args_()
    im_path = Path(args.im_path)
    msk_path = Path(args.msk_path)
    save_im_path = Path(args.save_im_path) if args.save_im_path is not None else None
    save_msk_path = Path(args.save_msk_path) if args.save_msk_path is not None else None
    patchify_path(
        im_path=im_path, 
        patch_size=args.patch_size, 
        save_im_path=save_im_path, 
        save_msk_path=save_msk_path, 
        msk_path=msk_path,
        min_pixel=args.min_pixel
        )

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('23_patch_image_128.ipynb')